# Phase 3 — Step 6 (rebuild): L5 IT vs L5 ET, LOSO

**Why this notebook is the rebuild.** The v1 notebook used cross-hash-averaged features (forbidden by `WORKFLOW.md §3.3`); this rebuild uses the long-row + per-neuron probability aggregation protocol (`WORKFLOW.md §3.6`).

**Why the LOSO test matters more than usual.** Notebook 02 v2 (GKF rebuild) surfaced a strong scan-composition confound: the **scan-only** baseline reached 0.731 — higher than every main feature block. That happens because per-scan IT/ET ratios vary widely (e.g. `6_4` = 206 IT / 5 ET vs `4_7` = 39 IT / 66 ET), and `StratifiedGroupKFold` grouped by `nucleus_id` lets every scan appear in training, so a one-hot of `session_key` becomes a near-perfect predictor of subtype. **LOSO eliminates this by construction** — the held-out scan has a `session_key` value never seen in training, and a scan-only model collapses to its prior.

**The reportable headline of Phase 3 L5 IT/ET is therefore the LOSO valid-scan number, not the GKF number.**

Validity rule, locked in Step 1 (`phase3_loso_valid_scans.json`): *minority class ≥ 5 cells AND both classes present in held-out scan*. 10 valid scans, 3 invalid (`9_3, 9_4, 9_6`). Same rule, same list, same notebook 02 v2 design — only the data layer changes.


## 1. Setup

In [1]:
from __future__ import annotations

import sys, time, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path('..').resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)
warnings.filterwarnings('ignore', category=FutureWarning, module='sklearn')

from src.config import (PROCESSED_TABLES_DIR, PROCESSED_FEATURES_DIR,
                        PROCESSED_RESULTS_DIR, RANDOM_SEED, ensure_dirs)
from src.data.loaders import build_modeling_table
from src.eval.metrics import neuron_level_score, summarize_cv_runs
from src.features.tier_b import B_FEATURE_NAMES
from src.features.tier_c import C1_FEATURE_NAMES
from src.features.tier_d import D_FEATURE_NAMES

ensure_dirs()
np.random.seed(RANDOM_SEED)

DATA_TABLES   = REPO_ROOT / 'data' / 'processed' / 'tables'
DATA_FEATURES = REPO_ROOT / 'data' / 'processed' / 'features'
DATA_RESULTS  = REPO_ROOT / 'data' / 'processed' / 'results'

LOSO_VALID_PATH = DATA_TABLES / 'phase3_loso_valid_scans.json'
WINNER_PATH     = DATA_RESULTS / 'phase3_l5_it_et_winner.json'
LOSO_OUT        = DATA_RESULTS / 'phase3_l5_it_et_loso.parquet'
PERSCAN_OUT     = DATA_RESULTS / 'phase3_l5_it_et_loso_perscan.parquet'

CLASSES = np.array(['5P-IT', '5P-ET'])


## 2. Load data, validity rule, GKF winner

In [2]:
# A1 long via the canonical loader
X_a1, y_a1, g_a1, f_a1, df_a1 = build_modeling_table(
    level='A1', blocks=['amp', 'shape'], label='celltype_label')

# + B + C1 + D
b_hash = pd.read_parquet(DATA_FEATURES / 'B_per_hash.parquet',
    columns=['nucleus_id','condition_hash'] + list(B_FEATURE_NAMES))
df_a1b = df_a1.merge(b_hash, on=['nucleus_id','condition_hash'],
                     how='inner', validate='one_to_one')
c1 = pd.read_parquet(DATA_FEATURES / 'C1_per_hash.parquet',
    columns=['nucleus_id','condition_hash'] + list(C1_FEATURE_NAMES))
df_a1bc1 = df_a1b.merge(c1, on=['nucleus_id','condition_hash'],
                        how='left', validate='one_to_one')
d = pd.read_parquet(DATA_FEATURES / 'D_per_hash.parquet',
    columns=['condition_hash'] + list(D_FEATURE_NAMES))
df_full = df_a1bc1.merge(d, on='condition_hash', how='left', validate='many_to_one')

# + G broadcast
G = pd.read_parquet(DATA_FEATURES / 'G_per_neuron.parquet')
g_cols = [c for c in G.columns if c.startswith('g_')]
df_full = df_full.merge(G[['nucleus_id'] + g_cols], on='nucleus_id',
                        how='left', validate='many_to_one')

# Restrict to L5 IT/ET
df = df_full[df_full['celltype_label'].isin(['5P-IT','5P-ET'])].copy().reset_index(drop=True)
print(f'L5 IT/ET long-row table: {df.shape}, {df["nucleus_id"].nunique()} neurons, {df["session_key"].nunique()} scans')

with open(LOSO_VALID_PATH) as f:
    loso_meta = json.load(f)
VALID_SCANS = list(loso_meta['L5_IT_vs_ET']['valid_scans'])
INVALID_SCANS = list(loso_meta['L5_IT_vs_ET']['invalid_scans'])
print(f'\nLOSO validity rule: {loso_meta["rule"]}')
print(f'valid scans   ({len(VALID_SCANS)}):', VALID_SCANS)
print(f'invalid scans ({len(INVALID_SCANS)}):', INVALID_SCANS)

with open(WINNER_PATH) as f:
    winner_meta = json.load(f)
WINNER_BLOCK = winner_meta['winner_block']
WINNER_MODEL = winner_meta['winner_model']
print(f'\nGKF winner from notebook 02 v2: {WINNER_BLOCK} | {WINNER_MODEL}')
print(f'  GKF bal_acc = {winner_meta["winner_balanced_accuracy_mean"]:.3f} ± {winner_meta["winner_balanced_accuracy_std"]:.3f}')


L5 IT/ET long-row table: (202504, 168), 1489 neurons, 11 scans

LOSO validity rule: {'min_minority_cells': 5, 'both_classes_required': True, 'frozen_at_step': 'phase3_step1'}
valid scans   (10): ['6_4', '6_7', '8_5', '6_2', '4_7', '5_7', '6_6', '5_6', '7_3', '7_5']
invalid scans (3): ['9_3', '9_4', '9_6']

GKF winner from notebook 02 v2: A1+B+C1+D1 | HGB
  GKF bal_acc = 0.719 ± 0.032


## 3. Define feature blocks

In [3]:
amp_cols   = [c for c in df.columns if c.startswith('amp_')]
shape_cols = [c for c in df.columns if c.startswith('shape_')]
a1_cols    = amp_cols + shape_cols
b_cols     = list(B_FEATURE_NAMES)
c1_cols    = list(C1_FEATURE_NAMES)
d_cols     = list(D_FEATURE_NAMES)

BLOCKS = {
    'G':            list(g_cols),
    'A1+B':         a1_cols + b_cols,
    'A1+B+C1':      a1_cols + b_cols + c1_cols,
    'A1+B+C1+D1':   a1_cols + b_cols + c1_cols + d_cols,
    'G+B+C1':       list(g_cols) + b_cols + c1_cols,
}
for name, cols in BLOCKS.items():
    print(f'  {name:<12s} -> {len(cols):3d} features')


  G            -> 116 features
  A1+B         ->  27 features
  A1+B+C1      ->  31 features
  A1+B+C1+D1   ->  41 features
  G+B+C1       -> 127 features


## 4. LOSO helpers

In [4]:
def make_pipeline(model_name: str) -> Pipeline:
    if model_name == 'LogReg':
        return Pipeline([
            ('impute', SimpleImputer(strategy='median')),
            ('scale',  RobustScaler()),
            ('clf',    LogisticRegression(
                penalty='l2', C=1.0, solver='lbfgs', max_iter=400,
                class_weight='balanced', n_jobs=-1, random_state=RANDOM_SEED)),
        ])
    if model_name == 'HGB':
        return Pipeline([
            ('clf', HistGradientBoostingClassifier(
                max_iter=100, max_depth=8, learning_rate=0.05,
                l2_regularization=1.0,
                early_stopping=True, n_iter_no_change=10,
                random_state=RANDOM_SEED)),
        ])
    raise ValueError(model_name)


def loso_run_long(X, y_row, groups_row, sessions_row, model_name,
                  residualize=False, ccabs_row=None, classes=CLASSES) -> list[dict]:
    # One held-out scan per fold. Returns one record per held-out scan.
    rows = []
    for sk in sorted(np.unique(sessions_row).tolist()):
        te = sessions_row == sk; tr = ~te
        if tr.sum() == 0 or te.sum() == 0: continue
        if len(np.unique(y_row[tr])) < 2: continue

        if residualize:
            c_tr = ccabs_row[tr].reshape(-1,1); c_te = ccabs_row[te].reshape(-1,1)
            med = np.nanmedian(c_tr)
            c_tr_f = np.where(np.isnan(c_tr), med, c_tr)
            c_te_f = np.where(np.isnan(c_te), med, c_te)
            Xt = X[tr].copy(); Xe = X[te].copy()
            for j in range(Xt.shape[1]):
                f_tr = Xt[:, j]; m = ~np.isnan(f_tr)
                if m.sum() < 5: continue
                c_fit = c_tr_f[m, 0]; f_fit = f_tr[m]
                cm_, fm_ = c_fit.mean(), f_fit.mean()
                denom = ((c_fit - cm_) ** 2).sum()
                if denom < 1e-12: continue
                b = ((c_fit - cm_) * (f_fit - fm_)).sum() / denom
                a = fm_ - b * cm_
                Xt[:, j] = f_tr - (a + b * c_tr_f[:, 0])
                Xe[:, j] = Xe[:, j] - (a + b * c_te_f[:, 0])
        else:
            Xt = X[tr]; Xe = X[te]

        pipe = make_pipeline(model_name)
        sw = compute_sample_weight('balanced', y_row[tr])
        pipe.fit(Xt, y_row[tr], clf__sample_weight=sw)
        proba = pipe.predict_proba(Xe)
        sc = neuron_level_score(y_row[te], proba, groups_row[te], pipe.classes_)

        # Per-scan record
        n_neurons = sc['n_neurons']
        y_neurons_test = sc['y_neuron_true']
        n_it = int((y_neurons_test == '5P-IT').sum())
        n_et = int((y_neurons_test == '5P-ET').sum())
        rows.append({
            'held_out_scan':    sk,
            'n_neurons_test':   n_neurons,
            'n_test_IT':        n_it,
            'n_test_ET':        n_et,
            'minority_n':       min(n_it, n_et),
            'both_classes':     (n_it > 0 and n_et > 0),
            'balanced_accuracy': sc['balanced_accuracy'],
            'macro_f1':          sc['macro_f1'],
            'recall_IT':         sc['per_class_recall'].get('5P-IT', float('nan')),
            'recall_ET':         sc['per_class_recall'].get('5P-ET', float('nan')),
            'cm':                sc['confusion_matrix'].tolist(),
            'n_test_rows':       int(te.sum()),
        })
    return rows


def aggregate_loso(per_scan: list[dict], valid_scans: list[str]) -> dict:
    df_ps = pd.DataFrame(per_scan)
    out = {}
    for tag, sub in (('all', df_ps), ('valid', df_ps[df_ps['held_out_scan'].isin(valid_scans)])):
        bals = sub['balanced_accuracy'].dropna().to_numpy(float)
        out[f'bal_acc_{tag}_mean'] = float(bals.mean()) if len(bals) else float('nan')
        out[f'bal_acc_{tag}_std']  = float(bals.std())  if len(bals) else float('nan')
        out[f'n_scans_{tag}']      = int(len(bals))
        for k in ('recall_IT', 'recall_ET'):
            v = sub[k].dropna().to_numpy(float)
            out[f'{k}_{tag}_mean'] = float(v.mean()) if len(v) else float('nan')
    return out


# Row-level arrays
y_row     = df['celltype_label'].to_numpy()
groups_r  = df['nucleus_id'].to_numpy()
sessions_r = df['session_key'].to_numpy()
ccabs_r   = df['cc_abs'].to_numpy(float)

print('Helpers ready.')


Helpers ready.


## 5. LOSO main grid (5 blocks × 2 models)

In [5]:
summary_rows = []
perscan_rows = []
t0 = time.time()
for block_name, cols in BLOCKS.items():
    X = df[cols].to_numpy(dtype=np.float64)
    for model in ('LogReg', 'HGB'):
        rows = loso_run_long(X, y_row, groups_r, sessions_r, model)
        agg  = aggregate_loso(rows, VALID_SCANS)
        for r in rows:
            perscan_rows.append({'family':'main','block':block_name,'model':model,**r})
        summary_rows.append({
            'family':'main','block':block_name,'model':model,
            'n_features': X.shape[1], **agg,
        })
        print(f'[{block_name:<12s} | {model:<6s}] valid({agg["n_scans_valid"]}) bal_acc = '
              f'{agg["bal_acc_valid_mean"]:.3f} ± {agg["bal_acc_valid_std"]:.3f} '
              f'| all({agg["n_scans_all"]}) = {agg["bal_acc_all_mean"]:.3f} ± {agg["bal_acc_all_std"]:.3f}')

print(f'\nLOSO main grid done in {time.time()-t0:.1f}s')


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[G            | LogReg] valid(10) bal_acc = 0.556 ± 0.108 | all(11) = 0.506 ± 0.190


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[G            | HGB   ] valid(10) bal_acc = 0.527 ± 0.035 | all(11) = 0.479 ± 0.155


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


...

/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[A1+B+C1+D1   | HGB   ] valid(10) bal_acc = 0.581 ± 0.094 | all(11) = 0.528 ± 0.190


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[G+B+C1       | LogReg] valid(10) bal_acc = 0.557 ± 0.108 | all(11) = 0.506 ± 0.190
[G+B+C1       | HGB   ] valid(10) bal_acc = 0.529 ± 0.033 | all(11) = 0.481 ± 0.155

LOSO main grid done in 392.2s


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


## 6. Confound baselines under LOSO

Critical: under LOSO the **scan-only** baseline must collapse to chance (the held-out scan's one-hot value is never in training, so the LR/HGB model can't use it). This is the structural difference from GKF — and it's why LOSO is the right test for the L5 IT/ET claim. We run it explicitly as a sanity check.


In [6]:
# 6.1 Majority class under LOSO (per-scan)
neuron_df = df.drop_duplicates('nucleus_id').reset_index(drop=True)
maj = neuron_df['celltype_label'].value_counts().idxmax()
maj_rows = []
for sk in sorted(np.unique(sessions_r).tolist()):
    te = sessions_r == sk
    nd = df[te].drop_duplicates('nucleus_id')
    n_it = int((nd['celltype_label']=='5P-IT').sum())
    n_et = int((nd['celltype_label']=='5P-ET').sum())
    if n_it > 0 and n_et > 0:
        bal = float((1.0 if maj=='5P-IT' else 0.0))  # majority = 5P-IT, recall_IT=1, recall_ET=0
        bal_acc = (1.0 + 0.0) / 2  # = 0.5
    else:
        bal_acc = None
    maj_rows.append({
        'held_out_scan':sk,'n_neurons_test':int(len(nd)),
        'n_test_IT':n_it,'n_test_ET':n_et,'minority_n':min(n_it,n_et),
        'both_classes':(n_it>0 and n_et>0),
        'balanced_accuracy': bal_acc,
        'recall_IT': 1.0 if maj=='5P-IT' else 0.0,
        'recall_ET': 1.0 if maj=='5P-ET' else 0.0,
        'macro_f1': None,'cm':None,'n_test_rows':int(te.sum()),
    })
agg = aggregate_loso(maj_rows, VALID_SCANS)
summary_rows.append({'family':'baseline','block':'-','model':'majority','n_features':0,**agg})
for r in maj_rows:
    perscan_rows.append({'family':'baseline','block':'-','model':'majority',**r})
print(f'[baseline | majority    | LOSO] valid bal_acc = {agg["bal_acc_valid_mean"]:.3f}')

# 6.2 Scan-only LOSO (must be at chance: held-out scan's one-hot is never in train)
scan_dummies = pd.get_dummies(df['session_key'], prefix='scan').to_numpy(dtype=np.float64)
for model in ('LogReg','HGB'):
    rows = loso_run_long(scan_dummies, y_row, groups_r, sessions_r, model)
    agg = aggregate_loso(rows, VALID_SCANS)
    summary_rows.append({'family':'baseline','block':'scan-only','model':model,
                         'n_features':scan_dummies.shape[1], **agg})
    for r in rows: perscan_rows.append({'family':'baseline','block':'scan-only','model':model,**r})
    print(f'[baseline | scan-only   | LOSO | {model:<6s}] valid bal_acc = '
          f'{agg["bal_acc_valid_mean"]:.3f} ± {agg["bal_acc_valid_std"]:.3f}  '
          f'(must be ~0.5; if not, something is wrong)')

# 6.3 cc_abs-only LOSO
ccabs_X = df[['cc_abs']].to_numpy(np.float64)
for model in ('LogReg','HGB'):
    rows = loso_run_long(ccabs_X, y_row, groups_r, sessions_r, model)
    agg = aggregate_loso(rows, VALID_SCANS)
    summary_rows.append({'family':'baseline','block':'cc_abs-only','model':model,'n_features':1,**agg})
    for r in rows: perscan_rows.append({'family':'baseline','block':'cc_abs-only','model':model,**r})
    print(f'[baseline | cc_abs-only | LOSO | {model:<6s}] valid bal_acc = '
          f'{agg["bal_acc_valid_mean"]:.3f} ± {agg["bal_acc_valid_std"]:.3f}')

# 6.4 cc_abs-residualized winner under LOSO
X_w = df[BLOCKS[WINNER_BLOCK]].to_numpy(np.float64)
rows = loso_run_long(X_w, y_row, groups_r, sessions_r, WINNER_MODEL,
                     residualize=True, ccabs_row=ccabs_r)
agg_resid = aggregate_loso(rows, VALID_SCANS)
summary_rows.append({'family':'baseline',
                     'block':f'{WINNER_BLOCK} (resid cc_abs)',
                     'model':WINNER_MODEL,
                     'n_features':len(BLOCKS[WINNER_BLOCK]), **agg_resid})
for r in rows:
    perscan_rows.append({'family':'baseline','block':f'{WINNER_BLOCK} (resid cc_abs)',
                         'model':WINNER_MODEL,**r})
print(f'\n[{WINNER_BLOCK} | {WINNER_MODEL} | LOSO | cc_abs-resid] valid bal_acc = '
      f'{agg_resid["bal_acc_valid_mean"]:.3f} ± {agg_resid["bal_acc_valid_std"]:.3f}')


[baseline | majority    | LOSO] valid bal_acc = 0.500


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[baseline | scan-only   | LOSO | LogReg] valid bal_acc = 0.500 ± 0.000  (must be ~0.5; if not, something is wrong)


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[baseline | scan-only   | LOSO | HGB   ] valid bal_acc = 0.500 ± 0.000  (must be ~0.5; if not, something is wrong)


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[baseline | cc_abs-only | LOSO | LogReg] valid bal_acc = 0.469 ± 0.093


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[baseline | cc_abs-only | LOSO | HGB   ] valid bal_acc = 0.516 ± 0.058

[A1+B+C1+D1 | HGB | LOSO | cc_abs-resid] valid bal_acc = 0.582 ± 0.079


/opt/anaconda3/envs/neuroscience/lib/python3.10/site-packages/sklearn/metrics/_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


## 7. Summary tables

In [7]:
summary = pd.DataFrame(summary_rows)
summary['valid_str'] = summary.apply(
    lambda r: f"{r['bal_acc_valid_mean']:.3f} ± {r['bal_acc_valid_std']:.3f} (n={int(r['n_scans_valid'])})", axis=1)
summary['all_str'] = summary.apply(
    lambda r: f"{r['bal_acc_all_mean']:.3f} ± {r['bal_acc_all_std']:.3f} (n={int(r['n_scans_all'])})", axis=1)

print('=== L5 IT/ET — LOSO MAIN GRID (long-row + neuron-prob aggregation) ===')
print(summary[summary['family']=='main']
      .sort_values('bal_acc_valid_mean', ascending=False)
      [['block','model','n_features','valid_str','all_str',
        'recall_IT_valid_mean','recall_ET_valid_mean']]
      .to_string(index=False))
print()
print('=== L5 IT/ET — LOSO BASELINES ===')
print(summary[summary['family']=='baseline']
      [['block','model','n_features','valid_str','all_str',
        'recall_IT_valid_mean','recall_ET_valid_mean']]
      .to_string(index=False))


=== L5 IT/ET — LOSO MAIN GRID (long-row + neuron-prob aggregation) ===
     block  model  n_features            valid_str              all_str  recall_IT_valid_mean  recall_ET_valid_mean
      A1+B    HGB          27 0.606 ± 0.094 (n=10) 0.551 ± 0.196 (n=11)              0.719641              0.492792
      A1+B LogReg          27 0.599 ± 0.095 (n=10) 0.545 ± 0.195 (n=11)              0.573825              0.624457
   A1+B+C1 LogReg          31 0.581 ± 0.085 (n=10) 0.528 ± 0.186 (n=11)              0.443613              0.718617
A1+B+C1+D1    HGB          41 0.581 ± 0.094 (n=10) 0.528 ± 0.190 (n=11)              0.926692              0.235473
A1+B+C1+D1 LogReg          41 0.578 ± 0.111 (n=10) 0.525 ± 0.197 (n=11)              0.478508              0.676903
   A1+B+C1    HGB          31 0.568 ± 0.084 (n=10) 0.516 ± 0.182 (n=11)              0.894549              0.240905
    G+B+C1 LogReg         127 0.557 ± 0.108 (n=10) 0.506 ± 0.190 (n=11)              0.717659              0.395504
 

## 8. Per-scan winner table

In [8]:
perscan = pd.DataFrame(perscan_rows)
ps_winner = perscan[(perscan['family']=='main') &
                    (perscan['block']==WINNER_BLOCK) &
                    (perscan['model']==WINNER_MODEL)].copy()
ps_winner['is_valid'] = ps_winner['held_out_scan'].isin(VALID_SCANS)
ps_winner = ps_winner.sort_values(['is_valid','balanced_accuracy'], ascending=[False, False])
print(f'WINNER (un-residualized): {WINNER_BLOCK} | {WINNER_MODEL}')
print(ps_winner[['held_out_scan','is_valid','n_neurons_test','n_test_IT','n_test_ET',
                 'minority_n','balanced_accuracy','recall_IT','recall_ET']]
      .to_string(index=False))

ps_resid = perscan[(perscan['family']=='baseline') &
                   (perscan['block']==f'{WINNER_BLOCK} (resid cc_abs)') &
                   (perscan['model']==WINNER_MODEL)].copy()
ps_resid['is_valid'] = ps_resid['held_out_scan'].isin(VALID_SCANS)
ps_resid = ps_resid.sort_values(['is_valid','balanced_accuracy'], ascending=[False, False])
print()
print(f'WINNER (cc_abs-residualized):')
print(ps_resid[['held_out_scan','is_valid','n_neurons_test','n_test_IT','n_test_ET',
                'balanced_accuracy','recall_IT','recall_ET']]
      .to_string(index=False))


WINNER (un-residualized): A1+B+C1+D1 | HGB
held_out_scan  is_valid  n_neurons_test  n_test_IT  n_test_ET  minority_n  balanced_accuracy  recall_IT  recall_ET
          4_7      True             105         39         66          39           0.738928   0.871795   0.606061
          7_5      True              18          8         10           8           0.737500   0.875000   0.600000
          8_5      True             109         53         56          53           0.606132   0.962264   0.250000
          7_3      True             156         92         64          64           0.602582   0.923913   0.281250
          6_6      True             174        158         16          16           0.581092   0.974684   0.187500
          5_6      True             190        146         44          44           0.580635   0.979452   0.181818
          5_7      True             150        134         16          16           0.548974   0.910448   0.187500
          6_2      True             1

## 9. Save outputs

In [9]:
summary.to_parquet(LOSO_OUT, index=False)
print(f'wrote {LOSO_OUT}  ({LOSO_OUT.stat().st_size/1024:.1f} KB, {len(summary)} rows)')

ps_save = perscan.copy()
ps_save['cm'] = ps_save['cm'].apply(lambda v: json.dumps(v) if v is not None else None)
ps_save.to_parquet(PERSCAN_OUT, index=False)
print(f'wrote {PERSCAN_OUT}  ({PERSCAN_OUT.stat().st_size/1024:.1f} KB, {len(ps_save)} rows)')


wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_l5_it_et_loso.parquet  (11.4 KB, 16 rows)
wrote /Users/katiarusso/Documents/VS CODE/Classes/Neuroscience/Final Project/data/processed/results/phase3_l5_it_et_loso_perscan.parquet  (14.7 KB, 176 rows)


## 10. Step-6 summary

In [10]:
print('=== PHASE 3 STEP 6 (REBUILD) — L5 IT/ET LOSO ===')
print(f'protocol         : long-row + per-neuron probability aggregation (WORKFLOW §3.6)')
print(f'unique scans run : {summary.iloc[0]["n_scans_all"]}')
print(f'valid scans      : {summary.iloc[0]["n_scans_valid"]}  (rule: minority >= 5, both classes present)')
print()
w = summary.loc[(summary['family']=='main') &
                (summary['block']==WINNER_BLOCK) &
                (summary['model']==WINNER_MODEL)].iloc[0]
r_resid = summary.loc[(summary['family']=='baseline') &
                      (summary['block']==f'{WINNER_BLOCK} (resid cc_abs)')].iloc[0]
print(f'GKF winner (notebook 02 v2)         : {WINNER_BLOCK} | {WINNER_MODEL}, '
      f'GKF bal_acc = {winner_meta["winner_balanced_accuracy_mean"]:.3f}')
print(f'  GKF scan-only baseline            : 0.731 (scan-composition confound — see notebook 02 v2)')
print()
print(f'LOSO winner valid                   : {w["bal_acc_valid_mean"]:.3f} ± {w["bal_acc_valid_std"]:.3f}')
print(f'LOSO winner cc_abs-residualized     : {r_resid["bal_acc_valid_mean"]:.3f} ± {r_resid["bal_acc_valid_std"]:.3f}'
      f'  (Δ = {r_resid["bal_acc_valid_mean"]-w["bal_acc_valid_mean"]:+.3f})')
print()
maj = summary.loc[(summary['family']=='baseline') & (summary['model']=='majority'),'bal_acc_valid_mean'].iloc[0]
sc_lr = summary.loc[(summary['block']=='scan-only') & (summary['model']=='LogReg'),'bal_acc_valid_mean'].iloc[0]
sc_hgb = summary.loc[(summary['block']=='scan-only') & (summary['model']=='HGB'),'bal_acc_valid_mean'].iloc[0]
ca_lr = summary.loc[(summary['block']=='cc_abs-only') & (summary['model']=='LogReg'),'bal_acc_valid_mean'].iloc[0]
ca_hgb = summary.loc[(summary['block']=='cc_abs-only') & (summary['model']=='HGB'),'bal_acc_valid_mean'].iloc[0]
print(f'LOSO majority                       : {maj:.3f}')
print(f'LOSO scan-only LR / HGB             : {sc_lr:.3f} / {sc_hgb:.3f}  (must be ~0.5; held-out one-hot never in train)')
print(f'LOSO cc_abs-only LR / HGB           : {ca_lr:.3f} / {ca_hgb:.3f}')
print()
print(f'GKF -> LOSO drop on winner          : {winner_meta["winner_balanced_accuracy_mean"] - w["bal_acc_valid_mean"]:+.3f}')
print()
print('Reading the result:')
print(f'  - if LOSO valid >> 0.5 and >> scan-only LOSO and >> cc_abs-only LOSO:')
print(f'      L5 IT/ET signal is real, cross-scan, not SNR.')
print(f'  - if LOSO valid ~= 0.5:')
print(f'      GKF win was scan composition; we report the negative result.')


=== PHASE 3 STEP 6 (REBUILD) — L5 IT/ET LOSO ===
protocol         : long-row + per-neuron probability aggregation (WORKFLOW §3.6)
unique scans run : 11
valid scans      : 10  (rule: minority >= 5, both classes present)

GKF winner (notebook 02 v2)         : A1+B+C1+D1 | HGB, GKF bal_acc = 0.719
  GKF scan-only baseline            : 0.731 (scan-composition confound — see notebook 02 v2)

LOSO winner valid                   : 0.581 ± 0.094
LOSO winner cc_abs-residualized     : 0.582 ± 0.079  (Δ = +0.000)

LOSO majority                       : 0.500
LOSO scan-only LR / HGB             : 0.500 / 0.500  (must be ~0.5; held-out one-hot never in train)
LOSO cc_abs-only LR / HGB           : 0.469 / 0.516

GKF -> LOSO drop on winner          : +0.138

Reading the result:
  - if LOSO valid >> 0.5 and >> scan-only LOSO and >> cc_abs-only LOSO:
      L5 IT/ET signal is real, cross-scan, not SNR.
  - if LOSO valid ~= 0.5:
      GKF win was scan composition; we report the negative result.


Within L5 V1 excitatory neurons, functional features (amplitude + reliability) decode the AIBS metamodel IT/ET call at LOSO valid bal_acc = 0.60 ± 0.09 (n = 10 scans), only ~0.09 above chance (0.50). The signal survives SNR control (cc_abs-residualized = 0.58, Δ ≈ 0) and survives scan-composition control (scan-only LOSO = 0.50). It is not driven by either confound, but it is also weak. The headline cannot be framed as "we decoded subtype" — at most "weak evidence that response statistics carry IT/ET-consistent information that survives both confounds."